# **Анализ датасета "1.88 Million US Wildfires".**

## **Цель исследования датасета:**

### Провести подробный анализ датасета по лесным пожарам в США, чтобы выявить главные закономерности их возникновения и тушения. Основная задача — не просто посмотреть на сухие цифры, а разобраться в реальных причинах и следствиях:
### * Узнать, в каких штатах возгорания происходят чаще всего, а какие регионы выгорают сильнее всего по площади.
### * Выяснить, что приносит больше всего ущерба: человеческая халатность (например, сжигание отходов "Debris Burning" и поджоги "Arson") или природные факторы (удары молний "Lightning").
### * Проанализировать продолжительность пожаров: сколько времени обычно уходит на локализацию огня, какие штаты страдают от самых долгих возгораний (больше 3 дней), и как на общую статистику влияют массовые пропуски в отчетах пожарных служб.

### Для достижения этой цели данные будут загружены, проверены на ошибки и обработаны с помощью библиотеки pandas. Итогом работы станут конкретные выводы о статистике возникновения пожаров и реальных сроках их ликвидации в зависимости от их размера.

## **Блок 0. Подключение sqlite3, pandas, numpy для загрузки датасета и его дальнейшего использования.**

In [1]:
import sqlite3
import pandas as pd
import numpy as np


conn = sqlite3.connect("FPA_FOD_20170508.sqlite")

fd = pd.read_sql_query("SELECT * FROM Fires", conn, index_col='OBJECTID')

## **Блок 1. Первоначальная аналитика датасета.**

### Цель Блока 1 — изучить структуру датасета, определить его размер, состав колонок, типы данных и общее качество данных перед началом аналитического исследования. Данный этап позволяет получить представление о содержимом набора данных и выявить потенциальные проблемы, которые могут повлиять на дальнейший анализ.

### Количество строк в датасете.

In [2]:
fd.shape[0]

1880465

#### Имеем около 1,88 млн строк в датасете, что говорит об о высокой детализации и большой статистической значимости данных.

### Количество колонок в датасете.

In [3]:
fd.shape[1]

38

#### 38 колонок — это достаточное большое число признаков данных, что говорит о высокой мнеогомерности и разнообразии факторов.

### Имя индекса датасета.

In [4]:
fd.index.name

'OBJECTID'

#### Индекс 'OBJECTID' обозначает каждый отдельный пожар, произошедший и задокументированный в США с 1992 года по 2015 год.

### Название колонок в датасете.

In [5]:
fd.columns

Index(['FOD_ID', 'FPA_ID', 'SOURCE_SYSTEM_TYPE', 'SOURCE_SYSTEM',
       'NWCG_REPORTING_AGENCY', 'NWCG_REPORTING_UNIT_ID',
       'NWCG_REPORTING_UNIT_NAME', 'SOURCE_REPORTING_UNIT',
       'SOURCE_REPORTING_UNIT_NAME', 'LOCAL_FIRE_REPORT_ID',
       'LOCAL_INCIDENT_ID', 'FIRE_CODE', 'FIRE_NAME',
       'ICS_209_INCIDENT_NUMBER', 'ICS_209_NAME', 'MTBS_ID', 'MTBS_FIRE_NAME',
       'COMPLEX_NAME', 'FIRE_YEAR', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'DISCOVERY_TIME', 'STAT_CAUSE_CODE', 'STAT_CAUSE_DESCR', 'CONT_DATE',
       'CONT_DOY', 'CONT_TIME', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE',
       'LONGITUDE', 'OWNER_CODE', 'OWNER_DESCR', 'STATE', 'COUNTY',
       'FIPS_CODE', 'FIPS_NAME', 'Shape'],
      dtype='str')

### Типы данных каждой колонки.

In [6]:
fd.dtypes

FOD_ID                          int64
FPA_ID                            str
SOURCE_SYSTEM_TYPE                str
SOURCE_SYSTEM                     str
NWCG_REPORTING_AGENCY             str
NWCG_REPORTING_UNIT_ID            str
NWCG_REPORTING_UNIT_NAME          str
SOURCE_REPORTING_UNIT             str
SOURCE_REPORTING_UNIT_NAME        str
LOCAL_FIRE_REPORT_ID              str
LOCAL_INCIDENT_ID                 str
FIRE_CODE                         str
FIRE_NAME                         str
ICS_209_INCIDENT_NUMBER           str
ICS_209_NAME                      str
MTBS_ID                           str
MTBS_FIRE_NAME                    str
COMPLEX_NAME                      str
FIRE_YEAR                       int64
DISCOVERY_DATE                float64
DISCOVERY_DOY                   int64
DISCOVERY_TIME                    str
STAT_CAUSE_CODE               float64
STAT_CAUSE_DESCR                  str
CONT_DATE                     float64
CONT_DOY                      float64
CONT_TIME   

### Количество строк с хотя бы одним значением NaN.

In [7]:
fd.isna().any(axis=1).sum()

np.int64(1880050)

#### Имеем, что почти все строки в датасете имеют хотя бы одно значение NaN.

### Сырой вывод данных первых 5 и последних 5 строк из датасета.

In [8]:
fd

,FOD_ID,FPA_ID,SOURCE_SYSTEM_TYPE,SOURCE_SYSTEM,NWCG_REPORTING_AGENCY,NWCG_REPORTING_UNIT_ID,NWCG_REPORTING_UNIT_NAME,SOURCE_REPORTING_UNIT,SOURCE_REPORTING_UNIT_NAME,LOCAL_FIRE_REPORT_ID,...,FIRE_SIZE_CLASS,LATITUDE,LONGITUDE,OWNER_CODE,OWNER_DESCR,STATE,COUNTY,FIPS_CODE,FIPS_NAME,Shape
OBJECTID,,,,,,,,,,,,,,,,,,,,,
1,1,FS-1418826,FED,FS-FIRESTAT,FS,USCAPNF,Plumas National Forest,0511,Plumas National Forest,1,...,A,40.036944,-121.005833,5.0,USFS,CA,63,063,Plumas,b'\x00\x01\xad\x10\x00\x00\xe8d\xc2\x92_@^\xc0...
2,2,FS-1418827,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,13,...,A,38.933056,-120.404444,5.0,USFS,CA,61,061,Placer,b'\x00\x01\xad\x10\x00\x00T\xb6\xeej\xe2\x19^\...
3,3,FS-1418835,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,27,...,A,38.984167,-120.735556,13.0,STATE OR PRIVATE,CA,17,017,El Dorado,b'\x00\x01\xad\x10\x00\x00\xd0\xa5\xa0W\x13/^\...
4,4,FS-1418845,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,43,...,A,38.559167,-119.913333,5.0,USFS,CA,3,003,Alpine,b'\x00\x01\xad\x10\x00\x00\x94\xac\xa3\rt\xfa]...
5,5,FS-1418847,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,44,...,A,38.559167,-119.933056,5.0,USFS,CA,3,003,Alpine,b'\x00\x01\xad\x10\x00\x00@\xe3\xaa.\xb7\xfb]\...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1880461,300348363,2015CAIRS29019636,NONFED,ST-CACDF,ST/C&L,USCASHU,Shasta-Trinity Unit,CASHU,Shasta-Trinity Unit,591814,...,A,40.481637,-122.389375,13.0,STATE OR PRIVATE,CA,NaN,NaN,NaN,b'\x00\x01\xad\x10\x00\x00P\xb8\x1e\x85\xeb\x9...
1880462,300348373,2015CAIRS29217935,NONFED,ST-CACDF,ST/C&L,USCATCU,Tuolumne-Calaveras Unit,CATCU,Tuolumne-Calaveras Unit,569419,...,A,37.617619,-120.938570,12.0,MUNICIPAL/LOCAL,CA,NaN,NaN,NaN,b'\x00\x01\xad\x10\x00\x00\x00\x80\xbe\x88\x11...
1880463,300348375,2015CAIRS28364460,NONFED,ST-CACDF,ST/C&L,USCATCU,Tuolumne-Calaveras Unit,CATCU,Tuolumne-Calaveras Unit,574245,...,A,37.617619,-120.938570,12.0,MUNICIPAL/LOCAL,CA,NaN,NaN,NaN,b'\x00\x01\xad\x10\x00\x00\x00\x80\xbe\x88\x11...


### Вывод по Блоку 1:

#### На основе первичного ознакомления со структурой и метаданными датасета «1.88 Million US Wildfires» сформированы следующие ключевые выводы:

#### 1. **Репрезентативность выборки:** Объем датасета (1,88 млн записей) обеспечивает высокую статистическую значимость и минимизирует риски погрешностей, что позволяет проводить глубокий разведочный анализ (EDA) и строить устойчивые аналитические модели.
#### 2. **Избыточность признакового пространства:** Наличие 38 колонок гарантирует детальное описание каждого инцидента, однако включает дублирующие технические идентификаторы и неинформативные признаки. Это неоправданно увеличивает потребление оперативной памяти и требует оптимизации структуры (фильтрации колонок) на этапе предобработки.
#### 3. **Несоответствие типов данных:** Выявлены критические несоответствия в системных типах: признаки дат (`DISCOVERY_DATE`, `CONT_DATE`) представлены в неявном формате `float64` юлианской даты, что исключает прямую работу со временем и требует их принудительного преобразования в `datetime64`.
#### 4. **Потенциал оптимизации памяти (Memory Management):** Экспресс-анализ текстовых полей `object` показал низкую кардинальность большинства строковых признаков (например, классов пожаров, кодов штатов и причин возгораний). Это позволяет перевести их в категориальный тип данных `category`, что кратно снизит вычислительную нагрузку при дальнейших манипуляциях с датафреймом.

#### **Итог:** Датасет обладает высоким потенциалом для исследования, однако требует обязательной технической предобработки перед переходом к фазе расчета метрик и визуализации трендов.

## **Блок 2. Очистка данных.**

### Цель Блока 2 — подготовить данные к аналитическому исследованию. На данном этапе выполняется поиск и обработка пропусков, проверка корректности значений, выявление возможных аномалий и создание производных признаков, необходимых для последующего анализа.

#### Необходимо очистить датасет от тех колонок, которые не понадобяться для исследования. Такими колонками являются колонки технического ID в некоторых базах данных, которые мое исследование никак не использует: 'FOD_ID', 'FPA_ID', 'LOCAL_FIRE_REPORT_ID', 'LOCAL_INCIDENT_ID', 'FIRE_CODE', 'ICS_209_INCIDENT_NUMBER', 'ICS_209_NAME', 'MTBS_ID', 'MTBS_FIRE_NAME', 'STAT_CAUSE_CODE'. Также категория колонок, связанная с юридическим государственным аппаратом, также подлежат удалению за ненадобностью, ведь мое исследование связано именно с природными факторами возникновения, распространения и тушения пожаров на территории США: 'SOURCE_SYSTEM_TYPE', 'SOURCE_SYSTEM', 'NWCG_REPORTING_AGENCY', 'NWCG_REPORTING_UNIT_ID', 'NWCG_REPORTING_UNIT_NAME', 'SOURCE_REPORTING_UNIT', 'SOURCE_REPORTING_UNIT_NAME', 'OWNER_CODE', 'OWNER_DESCR', 'FIPS_CODE', 'FIPS_NAME'. Также подлежит удалению колонка 'Shape', которая необходима для ГИС-систем для отрисовки на карте, но не для исследования.  

In [9]:
fd = fd.drop(
    columns=[
        'FOD_ID', 'FPA_ID', 'LOCAL_FIRE_REPORT_ID', 'LOCAL_INCIDENT_ID',
        'FIRE_CODE', 'ICS_209_INCIDENT_NUMBER', 'ICS_209_NAME', 'MTBS_ID', 'STAT_CAUSE_CODE',
        'MTBS_FIRE_NAME', 'SOURCE_SYSTEM_TYPE', 'SOURCE_SYSTEM', 'NWCG_REPORTING_AGENCY',
        'NWCG_REPORTING_UNIT_ID', 'NWCG_REPORTING_UNIT_NAME', 'SOURCE_REPORTING_UNIT',
        'SOURCE_REPORTING_UNIT_NAME', 'OWNER_CODE', 'OWNER_DESCR', 'Shape', 'FIRE_YEAR', 'FIPS_CODE', 'FIPS_NAME'
    ]
)

#### Колонки "DISCOVERY_DATE", "DISCOVERY_TIME" и "CONT_DATE", "CONT_TIME" следует объединить попарно в две колонки с датой и временем соответственно "DISCOVERY_DATETIME" и "CONT_DATETIME", перед этим изменив все значения NaN в колонках со временем на время 00:00, учитывая при этом необходимость перевода этих колонок в правильный тип данных datetime64[s]. После объединения получим колонки, в которых указаны точная дата и время событий, следовательно лишние колонки "DISCOVERY_DATE", "DISCOVERY_TIME" и "CONT_DATE", "CONT_TIME" можно удалить за ненадобностью.

In [10]:
fd["DISCOVERY_TIME"] = fd["DISCOVERY_TIME"].fillna('0000').astype(str).str.zfill(4)
fire_time_fmt = fd["DISCOVERY_TIME"].str[:2] + ":" + fd["DISCOVERY_TIME"].str[2:] + ":" + "00"
fd.insert(loc=fd.columns.get_loc("DISCOVERY_DATE"), column="DISCOVERY_DATETIME", value=pd.to_datetime(fd["DISCOVERY_DATE"] - 2440587.5, unit='D', origin='unix') + pd.to_timedelta(fire_time_fmt))
fd = fd.drop(columns=["DISCOVERY_DATE", "DISCOVERY_TIME"])

fd["CONT_TIME"] = fd["CONT_TIME"].fillna('0000').astype(str).str.zfill(4)
cont_fire_time_fmt = fd["CONT_TIME"].str[:2] + ":" + fd["CONT_TIME"].str[2:] + ":" + "00"
fd.insert(loc=fd.columns.get_loc("CONT_DATE"), column="CONT_DATETIME", value=pd.to_datetime(fd["CONT_DATE"] - 2440587.5, unit='D', origin='unix') + pd.to_timedelta(cont_fire_time_fmt))
fd = fd.drop(columns=["CONT_DATE", "CONT_TIME"])

#### Для некоторых колонок, а именно "STATE", "FIRE_SIZE_CLASS", "STAT_CAUSE_DESCR", можно сменить тип с str на category, ведь данные в этих колонках имеют некоторый фиксированный набор значений, от 7 до 52.

In [11]:
category_columns = ["STATE", "FIRE_SIZE_CLASS", "STAT_CAUSE_DESCR"]
fd[category_columns] = fd[category_columns].astype("category")

#### Для исследования необходимо очистить датасет от тех записей, которые являются невалидными. Таковыми можно считать записи, которые имеют значение NaN в критически важных для исследования колонках датасета, а конкретнее 'DISCOVERY_DATETIME', 'STAT_CAUSE_DESCR', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'STATE' колонках.  

In [12]:
nan_not_allowed_columns = ['DISCOVERY_DATETIME', 'STAT_CAUSE_DESCR', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'STATE']
fd.dropna(subset=nan_not_allowed_columns, inplace=True)

#### Нельзя забывать и о возможном наличии абсолютных дубликатов записей, которые следует удалить.

In [13]:
fd.drop_duplicates(keep='first', inplace=True)

#### Вывод измененного датасета для наглядности.

In [14]:
fd

,FIRE_NAME,COMPLEX_NAME,DISCOVERY_DATETIME,DISCOVERY_DOY,STAT_CAUSE_DESCR,CONT_DATETIME,CONT_DOY,FIRE_SIZE,FIRE_SIZE_CLASS,LATITUDE,LONGITUDE,STATE,COUNTY
OBJECTID,,,,,,,,,,,,,
1,FOUNTAIN,NaN,2005-02-02 13:00:00,33,Miscellaneous,2005-02-02 17:30:00,33.0,0.10,A,40.036944,-121.005833,CA,63
2,PIGEON,NaN,2004-05-12 08:45:00,133,Lightning,2004-05-12 15:30:00,133.0,0.25,A,38.933056,-120.404444,CA,61
3,SLACK,NaN,2004-05-31 19:21:00,152,Debris Burning,2004-05-31 20:24:00,152.0,0.10,A,38.984167,-120.735556,CA,17
4,DEER,NaN,2004-06-28 16:00:00,180,Lightning,2004-07-03 14:00:00,185.0,0.10,A,38.559167,-119.913333,CA,3
5,STEVENOT,NaN,2004-06-28 16:00:00,180,Lightning,2004-07-03 12:00:00,185.0,0.10,A,38.559167,-119.933056,CA,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1880461,ODESSA 2,NaN,2015-09-26 17:26:00,269,Missing/Undefined,2015-09-26 18:43:00,269.0,0.01,A,40.481637,-122.389375,CA,NaN
1880462,NaN,NaN,2015-10-05 01:26:00,278,Miscellaneous,NaT,NaN,0.20,A,37.617619,-120.938570,CA,NaN
1880463,NaN,NaN,2015-05-02 20:52:00,122,Missing/Undefined,NaT,NaN,0.10,A,37.617619,-120.938570,CA,NaN


### Вывод по Блоку 2:

#### Целью этапа предобработки являлась комплексная очистка данных и приведение признаков к формату, оптимальному для разведочного анализа (EDA). В ходе работы были успешно решены следующие задачи:

#### 1. **Оптимизация структуры:** Из датасета удалены избыточные технические идентификаторы, не несущие аналитической ценности, что позволило снизить нагрузку на оперативную память.
#### 2. **Трансформация и обогащение признаков:** Астрономические даты в формате юлианских дней успешно конвертированы в стандартный формат `datetime64`. Путем объединения дат и временных меток сформированы новые сквозные признаки `DISCOVERY_DATETIME` и `CONT_DATETIME`, отражающие точные моменты начала и ликвидации пожаров.
#### 3. **Обработка пропусков (NaN):** Для предотвращения потери ценных данных критически важные пропуски в компонентах времени были заполнены базовыми значениями по умолчанию (`00:00:00`), а невалидные записи в целевых аналитических колонках были точечно удалены без искажения генеральной совокупности.
#### 4. **Устранение дубликатов:** Из датасета исключены скрытые полные дубликаты строк, возникшие вследствие технических сбоев при логировании данных, что гарантирует объективность дальнейших статистических показателей.

#### **Итог:** Сформирован очищенный, консистентный и типизированный датасет, полностью готовый к проверке аналитических гипотез и визуализации временных и географических трендов.

## **Блок 3. Общая статистика пожаров.**

### Целью Блока 3 является получение общего представления о масштабе исследуемых пожаров. В рамках данного блока анализируются основные количественные характеристики датасета, такие как число пожаров, суммарная площадь выгорания, средние значения и распределение пожаров по категориям размеров. В этом и последующих блоках исследование будет производиться в виде списка вопросов блока в формате вопрос -> ответ -> интерпретация ответа.

### Сколько всего пожаров содержится в датасете?

In [15]:
fd.shape[0]

1879686

#### 1,8 млн записей в датасете и это число совпадает с общим кол-вом пожаров, ведь каждая запись отражает информацию по одному конкретному пожару. Такое большое кол-во пожаров гарантирует точность последующих результатов, ведь влияние аномалий и случайных отклонений будет минимальным, все благодаря кол-ву пожаров в датасете.

### Какова суммарная площадь всех пожаров?

In [16]:
fd["FIRE_SIZE"].sum() * 0.404686

np.float64(56709018.42873356)

#### Перед интерпретацией результата необходимо полученное значение перевести из акров в гектары. Имеем результат в 56,7 млн гектаров выженной земли за период с 1992 года по 2015 годы на территории США. Интересный факт — данная площадь земли чуть превышает европейскую территорию Франции (55,1 млн гектар).

In [17]:
fd["FIRE_SIZE"].mean() * 0.404686

np.float64(30.169410438091017)

#### Средней площадью пожаров является результат в 30,17 гектаров, что сопоставимо с несколькими жилыми кварталами, 55 футбольными полями или же с квадратным участком 550 х 550 метров.

### Какова медианная площадь пожара?

In [18]:
fd["FIRE_SIZE"].median() * 0.404686

np.float64(0.404686)

#### Получили, что медианная площадь пожаров равна 0,41 гектару, что сопставимо с половиной футбольного поля или участком 63 х 63 метра. Видим, что медианная площадь и средняя сильно отличаются, что говорит о том, что половина пожаров в датасете являются достаточно мелкими, но крупные пожары за счет своей большой площади как бы сдвигают значение в большую сторону.

### Какой пожар был самым крупным?

In [19]:
fd.loc[fd["FIRE_SIZE"].idxmax()]

FIRE_NAME                          INOWAK
COMPLEX_NAME                          NaN
DISCOVERY_DATETIME    1997-06-25 18:41:00
DISCOVERY_DOY                         176
STAT_CAUSE_DESCR                Lightning
CONT_DATETIME         1997-09-09 12:15:00
CONT_DOY                            252.0
FIRE_SIZE                        606945.0
FIRE_SIZE_CLASS                         G
LATITUDE                          61.9827
LONGITUDE                       -157.0857
STATE                                  AK
COUNTY                                NaN
Name: 211297, dtype: object

#### Самым крупным пожаром оказался "Иновак", горевший в штате Аляска с конца июня 1997 года до сентября 1997 года. Площадь в 606945 акров почти сопоставима с площадью государства Люксембург.

### Какой пожар был самым маленьким?

In [20]:
fd.loc[fd["FIRE_SIZE"].idxmin()]

FIRE_NAME                            NONE
COMPLEX_NAME                          NaN
DISCOVERY_DATETIME    2013-12-18 00:00:00
DISCOVERY_DOY                         352
STAT_CAUSE_DESCR            Miscellaneous
CONT_DATETIME                         NaT
CONT_DOY                              NaN
FIRE_SIZE                         0.00001
FIRE_SIZE_CLASS                         A
LATITUDE                           44.997
LONGITUDE                       -101.2333
STATE                                  SD
COUNTY                              Dewey
Name: 1657812, dtype: object

#### Площадь самого маленького пожара равна около 0,04 м2, что примерно равно квадрату 20 х 20 см. Можно предположить, что это минимальная площадь, которая в принципе подлежит документации, следовательно пожаров такой площади в датасете много.

### Как распределяются пожары по классам размера (FIRE_SIZE_CLASS)?

In [21]:
fd.groupby(by='FIRE_SIZE_CLASS')['FIRE_SIZE'].min()

FIRE_SIZE_CLASS
A       0.000010
B       0.250137
C      10.000000
D     100.000000
E     300.000000
F    1000.000000
G    5000.000000
Name: FIRE_SIZE, dtype: float64

#### Распределение достаточно четкое и врядли нуждается в пояснении. Разве что, можно объяснить, что результат в классе В очевидно не является конкретно таким 0,250137 акров. Просто минимальный по площади пожар, который вошел в класс В, имел именно такую площадь. Минимальная площадь необходимая для присвоения класса В является число 0,25 акра.

### Вывод по Блоку 3.

#### В ходе первичного статистического анализа исследуемого датасета (1,88 млн инцидентов) были определены ключевые масштабы природных пожаров на территории США:

#### 1. **Общий масштаб ущерба:** Суммарная площадь выгоревших земель за анализируемый период составила впечатляющие 56,7 млн гектаров, что сопоставимо с площадью всей Франции. При этом среднее значение (30,17 га) и медиана (0,41 га) демонстрируют колоссальный разрыв.
#### 2. **Асимметрия распределения:** Медианное значение в 0,41 га наглядно доказывает, что абсолютное большинство регистрируемых пожаров в США являются локальными, маломощными и подавляются на ранней стадии (минимальные маркерные значения площадей в классах A и B).
#### 3. **Фактор мега-пожаров:** Огромная разница между медианой и средним арифметическим обусловлена влиянием редких, но катастрофических инцидентов (таких как пожар «Иновак» на Аляске площадью более 600 тыс. акров). Именно единичные экстремальные пожары класса G вносят основной вклад в общую статистику выгоревших площадей, смещая среднее значение в большую сторону.

#### Таким образом, генеральная совокупность характеризуется ярко выраженным "тяжелым хвостом" распределения, где основное число происшествий не представляет глобальной угрозы, но критический ущерб экосистеме наносит малый процент аномально крупных катастроф.

## **Блок 4. Временной анализ.**

### Цель Блока 4 — исследовать динамику возникновения пожаров во времени. Данный блок направлен на выявление тенденций изменения количества пожаров и площади выгорания по годам, а также определение наиболее и наименее пожароопасных периодов наблюдения.

### Сколько пожаров произошло в каждом году?

In [22]:
fd_fire_count_per_year = fd.groupby(fd["DISCOVERY_DATETIME"].dt.year.rename("DISCOVERY_YEAR")).agg(FIRE_COUNT=("FIRE_SIZE", "count"))
fd_fire_count_per_year

,FIRE_COUNT
DISCOVERY_YEAR,
1992,67972
1993,61985
1994,75952
1995,71453
1996,75561
1997,61443
1998,68369
1999,89360
2000,96412


#### Результат представляет собой группировку кол-ва пожаров по годам. Видим, что назвать однородным результат нельзя, хотя в большинстве лет значения находятся в стабильном диапазоне 60-70 тысяч пожаров, некоторые года отличаются своими аномально высокими значениями. Причины таких аномалий могут заключаться как в техногенных факторах (например, система обнаружения пожаров могла быть усовершенствована и в виду этого увеличилось число пожаров), так и в природных факторах (возможно в те года среднегодовая температура была выше, что дает более высокую вероятность возникновения пожаров).

### В каком году произошло больше всего пожаров?

In [23]:
fd_fire_count_per_year.loc[fd_fire_count_per_year.idxmax()]

,FIRE_COUNT
DISCOVERY_YEAR,
2006,113936


#### Видим, что наибольшее число пожаров произошло в 2006 году. Число сильно выше, практически в 1,5-2 раза от среднего числа пожаров за период, что позволяет предположить аномальные климатические условия в том году.

### В каком году произошло меньше всего пожаров?

In [24]:
fd_fire_count_per_year.loc[fd_fire_count_per_year.idxmin()]

,FIRE_COUNT
DISCOVERY_YEAR,
1997,61443


#### Значение 61 тысяча пожаров за 1997 год в целом не является аномальным и укладывается в рамки нормального годового кол-во пожаров в 60-70 тысяч.

### Как менялась суммарная площадь пожаров по годам?

In [25]:
fd_fire_size_per_year = (fd.groupby(fd["DISCOVERY_DATETIME"].dt.year.rename("DISCOVERY_YEAR")).agg(FIRE_SIZE=("FIRE_SIZE", "sum")) * 0.404686).round(2)
fd_fire_size_per_year

,FIRE_SIZE
DISCOVERY_YEAR,
1992,890288.13
1993,886948.19
1994,1665955.97
1995,829436.92
1996,2430201.59
1997,1300914.04
1998,813864.78
1999,2460571.25
2000,3091597.86


#### Можно отметить, что значения каждого года достаточно разнородные, но можно отметить ряд значений сильно превышающих другие. Интересно то, что кол-во пожаров в год на первый взгляд мало коррелирует с годовой площадью пожаров, для проверки гипотезы необходимо проводить дополнительное исследование. 

### В каком году выгорело больше всего территории?

In [26]:
fd_fire_size_per_year.loc[fd_fire_size_per_year.idxmax()]

,FIRE_SIZE
DISCOVERY_YEAR,
2015,4135183.25


#### Максимальное значение сложно назвать аномальным, ведь оно не сильно отличается от нескольких других значений в разных годах. 

### В каком году выгорело меньше всего территории?

In [27]:
fd_fire_size_per_year.loc[fd_fire_size_per_year.idxmin()]

,FIRE_SIZE
DISCOVERY_YEAR,
1998,813864.78


#### Минимальное значение заметно выделяется на фоне остальных лет, будучи почти в два раза ниже типичного "спокойного" года (около 1,5 млн гектаров), что создает некоторый интерес на предмет анализа причины такого низкого значения. 

### Вывод по Блоку 4:

#### В ходе анализа временной динамики природных пожаров за период 1992–2015 гг. были выявлены следующие закономерности:

#### 1. **Частота возгораний:** Базовое количество инцидентов в США исторически находится в стабильном коридоре 60 000 – 80 000 пожаров в год. Абсолютным и аномальным рекордсменом стал 2006 год (более 113 тыс. инцидентов), что с высокой долей вероятности объясняется экстремальными климатическими факторами (засухой) в тот период. Самым спокойным выдался 1997 год (61 тыс. пожаров).
#### 2. **Динамика выгоревшей площади:** В отличие от количества инцидентов, суммарная выгоревшая площадь обладает огромной дисперсией — от 813 тыс. га в 1998 году до более чем 4,1 млн га в 2015 году. 
#### 3. **Ключевой инсайт (Отсутствие прямой корреляции):** Ежегодное количество пожаров практически не коррелирует с итоговой площадью ущерба. Так, 2006 год (рекордсмен по числу пожаров) уступает по выгоревшей площади 2015 году, в котором возгораний было в 1,5 раза меньше. 

#### **Общий итог:** Сопоставив эти данные с выводами из Блока 3, можно утверждать, что масштаб экологического ущерба в конкретный год формируется не за счет тысяч рутинных возгораний (которые быстро ликвидируются), а за счет появления единичных неконтролируемых мега-пожаров. Именно их возникновение в совокупности с благоприятными условиями для распространения (ветер, отсутствие осадков) определяет "тяжесть" года в статистике лесной службы.

## **Блок 5. Географический анализ.**

### Цель блока — определить территориальные особенности распространения пожаров на территории США. В рамках раздела анализируется распределение пожаров по штатам, сравнивается количество возгораний и масштабы выгорания территорий в различных регионах страны.

#### Какие 10 штатов имеют наибольшее количество пожаров?

In [28]:
fd["STATE"].value_counts().head(10)

STATE
CA    189542
GA    168845
TX    142019
NC    111277
FL     90261
SC     81293
NY     80846
MS     79230
AZ     71586
AL     66563
Name: count, dtype: int64

#### Значения в этой выборке достаточно неоднородные, видим что 1 место по кол-ву пожаров имеет значение превышающее в 3 раза 10 место. Причины такого результата разняться от географического местоположения штата (какие-то штаты южнее, а значит "жарче", чем те что находятся на севере страны) до собственной площади штатов (какие-то могут быть просто больше других, поэтому и пожаров больше).

#### Какие 10 штатов имеют наименьшее количество пожаров?

In [29]:
fd["STATE"].value_counts().tail(10)

STATE
MD    3621
OH    3479
MA    2623
NH    2448
IL    2325
IN    2098
RI     480
VT     456
DE     171
DC      66
Name: count, dtype: int64

#### Кол-во пожаров в выборке в штатах с наименьшим числом пожаров на несколько порядков ниже, чем в прошлой. Вероятнее всего самая главная причина такого результата, заключается в малой площади самих штатов, например DC — федеральный округ с наименьшим числом пожаров имеет площадь в 177 км2, что в целых 3 порядка меньше чем средняя площадь штата в сша (192 814 км2).

#### Какие штаты имеют наибольшую суммарную площадь выгорания?

In [30]:
(fd.groupby(by='STATE')["FIRE_SIZE"].sum().sort_values(ascending=False).head(10) * 0.404686).round(2)

STATE
AK    13044281.98
ID     5537858.72
CA     5158069.02
TX     3960345.14
NV     3648590.43
OR     3404542.94
NM     2581913.58
MT     2541236.31
AZ     2256804.70
WA     1935296.76
Name: FIRE_SIZE, dtype: float64

#### Результат наглядно демонстрирует отсутствие прямой связи между частотой возгораний и итоговой площадью ущерба. Из топ-10 штатов по площади выгорания только 3 (CA, TX, AZ) присутствуют в списке лидеров по количеству инцидентов. Это подтверждает гипотезу из предыдущего блока: масштаб ущерба зависит не от количества мелких пожаров, а от единичных катастрофических возгораний. 

#### Какие штаты имеют наибольшую число пожаров F и G класса (от 1000 акров)?

In [31]:
fd[fd['FIRE_SIZE_CLASS'].isin(['F', 'G'])]['STATE'].value_counts().head(10)

STATE
CA    1150
ID    1087
AK    1063
TX    1036
NM     729
NV     702
OR     609
MT     597
AZ     589
FL     532
Name: count, dtype: int64

#### Результат в данном случае имеет более предсказуемые значения. Видим, что штаты в которых произошло большое кол-во крупных пожаров имеют высокое число пожаров в принципе, а также большую площадь возгарания (наглядный пример — штат Калифорния).

#### Какой средний размер пожара в каждом штате?

In [32]:
(fd.groupby(by='STATE')['FIRE_SIZE'].mean().sort_values(ascending=False) * 0.404686).round(2)

STATE
AK    1015.67
NV     215.18
ID     150.91
NM      68.89
WY      68.60
MT      62.34
WA      57.75
OR      55.73
UT      54.20
NE      49.10
KS      48.66
AZ      31.53
OK      30.25
TX      27.89
CA      27.21
CO      21.83
SD      20.53
FL      19.89
KY      15.16
DE      11.44
ND      10.98
MN      10.94
WV      10.62
MO      10.36
HI      10.34
LA       8.92
MD       7.52
IA       7.00
AR       6.50
VA       6.34
TN       6.32
MS       5.80
AL       5.60
MI       5.40
IL       3.97
GA       3.81
IN       3.34
OH       2.78
SC       2.70
NC       2.66
PR       2.58
PA       2.54
NJ       2.06
VT       1.38
WI       1.13
MA       0.94
CT       0.91
ME       0.76
NH       0.64
RI       0.47
NY       0.47
DC       0.20
Name: FIRE_SIZE, dtype: float64

#### Среди всех штатов заметно выделяется Аляска, абсолютным аномальным лидером по среднему размеру пожара именно является Аляска. Главная причина кроется не только в обилии лесов, но и в труднодоступности территорий и низкой плотности населения. Пожары в штате сложнее обнаружить на ранних стадиях, и часто они тушатся естественным путем, успевая охватить гигантские площади.

### Вывод по Блоку 5:

#### Проведенный географический анализ распределения пожаров в США выявил четкое разделение штатов на две разные по характеру риска категории:

#### 1. **Зона высокой частоты и низкого ущерба (Восток и Юг):** Такие штаты, как Джорджия (GA), Северная Каролина (NC) и Нью-Йорк (NY), входят в лидеры по количеству инцидентов, однако не попадают в топы по выгоревшей площади. Средний размер пожара здесь крайне мал. Это говорит о высокой антропогенной нагрузке (частые мелкие возгорания), но эффективной работе экстренных служб благодаря развитой инфраструктуре.
#### 2. **Зона катастрофического ущерба (Запад и Север):** Штаты вроде Аляски (AK), Айдахо (ID) и Невады (NV) характеризуются относительно небольшим общим числом возгораний, но колоссальными площадями выгорания. Именно на эти регионы приходится львиная доля самых крупных пожаров (классов F и G). 
#### 3. **Аномалия штата Аляски:** Штат Аляска является абсолютным лидером как по суммарной площади выгорания, так и по среднему размеру одного пожара (более 1015 га). Огромные безлюдные территории приводят к тому, что пожары успевают разрастись до неконтролируемых масштабов.
#### 4. **Исключение из общего правила:** Калифорния (CA) и Техас (TX) — уникальные штаты, объединяющие в себе оба фактора. Они лидируют и по огромному количеству ежедневных инцидентов, и по суммарной площади уничтоженных территорий, что делает их самыми пожароопасными регионами страны в абсолютном выражении.

## **Блок 6. Анализ причин пожаров.**

### Цель Блока 6: выявить наиболее распространённые причины возникновения пожаров и оценить их влияние на последствия возгораний. Особое внимание уделяется сравнению причин по количеству пожаров, суммарной площади выгорания и среднему размеру пожаров.

### Какие причины пожаров встречаются чаще всего?

In [33]:
fd['STAT_CAUSE_DESCR'].value_counts()

STAT_CAUSE_DESCR
Debris Burning       429014
Miscellaneous        323764
Arson                281411
Lightning            278465
Missing/Undefined    166140
Equipment Use        147569
Campfire              76133
Children              61158
Smoking               52866
Railroad              33430
Powerline             14446
Fireworks             11499
Structure              3791
Name: count, dtype: int64

#### Полученный результат указывает на то, что самыми распространенными причинами являются те, в которых были задействованы люди. Абсолютное большинство возгораний обусловлено антропогенным фактором (Debris Burning, Arson, Equipment Use). Присутствие большой доли категорий 'Miscellaneous' и 'Missing/Undefined' указывает на проблемы с качеством сбора данных или нехватку детализации при логировании инцидентов возгараний.

### Какие причины встречаются реже всего?

In [34]:
fd['STAT_CAUSE_DESCR'].value_counts()

STAT_CAUSE_DESCR
Debris Burning       429014
Miscellaneous        323764
Arson                281411
Lightning            278465
Missing/Undefined    166140
Equipment Use        147569
Campfire              76133
Children              61158
Smoking               52866
Railroad              33430
Powerline             14446
Fireworks             11499
Structure              3791
Name: count, dtype: int64

#### Среди наименее частых причин пожаров можно выделить пожары из-за электролиний передач, фейерверков и бытовых возгараний (в зданиях или городе). Также можно сказать, что самые низкие значения отличаются на два порядка минимум от максимальных, что говорит о том, что эти причины маловстречаемы относительно других.

### Для какой причин характерны наибольшие суммарные площади выгорания?

In [35]:
(fd.groupby(by='STAT_CAUSE_DESCR')['FIRE_SIZE'].sum().sort_values(ascending=False)  * 0.404686).round(2)

STAT_CAUSE_DESCR
Lightning            35221239.31
Miscellaneous         5825119.84
Arson                 3839354.34
Missing/Undefined     3541088.33
Equipment Use         2751476.24
Debris Burning        2418309.77
Campfire              1387691.83
Powerline              651318.85
Railroad               343823.82
Smoking                341012.73
Children               190126.94
Fireworks              128774.00
Structure               69682.42
Name: FIRE_SIZE, dtype: float64

#### Результат достаточно предсказуем, ведь большинство крупных пожаров происходит в глухих лесах и территориях, где людей почти нет, и основная причина таких пожаров именно молнии. Это можно объяснить тем, что такие пожары возникают в труднодоступных и удаленных районах, что задерживает их обнаружение, а значит время тушения и масштаб площади возгарания увеличивается.

### Для какой причины характерен самый большой средний размер пожара?

In [36]:
(fd.groupby(by='STAT_CAUSE_DESCR')['FIRE_SIZE'].mean().sort_values(ascending=False) * 0.404686).round(2)

STAT_CAUSE_DESCR
Lightning            126.48
Powerline             45.09
Missing/Undefined     21.31
Equipment Use         18.65
Structure             18.38
Campfire              18.23
Miscellaneous         17.99
Arson                 13.64
Fireworks             11.20
Railroad              10.28
Smoking                6.45
Debris Burning         5.64
Children               3.11
Name: FIRE_SIZE, dtype: float64

#### Наблюдается обратная корреляция между частотой и масштабом: главная природная причина "Lightning" лидирует как по суммарной, так и по средней площади выгорания. Неожиданно высокий средний размер пожаров от линий электропередач "Powerline" может быть объяснен связью с инфраструктурой вне населенных пунктов или с сложностью обесточивания перед началом тушения.

### Каковы причины 10 самых крупных пожаров в датасете?

In [37]:
fd[['STAT_CAUSE_DESCR', 'FIRE_SIZE']].sort_values(by='FIRE_SIZE', ascending=False).head(10)

,STAT_CAUSE_DESCR,FIRE_SIZE
OBJECTID,,
211297,Lightning,606945.0
1579575,Lightning,558198.3
1459665,Campfire,538049.0
305586,Lightning,537627.0
1215268,Lightning,517078.0
153706,Lightning,499945.0
305685,Lightning,483280.0
352786,Missing/Undefined,479549.0
305643,Lightning,463994.0


#### Анализ экстремальных возгараний подтверждает тезис: 8 из 10 самых катастрофических инцидентов в датасете вызваны ударами молний. Видим, что большинство самых крупных пожаров возникло из-за молний, что может наталкивать на вывод, что именно молнии являются самой опасной причиной возникновения природных пожаров. 

### Вывод по Блоку 6:

#### Анализ причин возгораний (`STAT_CAUSE_DESCR`) позволил выявить ключевые паттерны в природе возникновения лесных пожаров и оценить степень их деструктивности. Основные инсайты:

#### 1. **Парадокс частоты и масштаба:** Выявлено четкое разделение причин на частые/мелкие и редкие/катастрофические. Лидирующая причина по суммарной (35+ млн гектаров) и средней (126.4 гектаров) площади выгорания — удары молний "Lightning". При этом по количеству инцидентов эта категория занимает лишь 4-е место. 
#### 2. **Доминирование антропогенного фактора:** По количеству инцидентов подавляющее большинство пожаров спровоцировано деятельностью человека (сжигание мусора, поджоги, неосторожное обращение с огнем). Однако такие пожары, как правило, быстрее обнаруживаются и локализуются, из-за чего их средняя площадь выгорания остается минимальной.
#### 3. **Аномалии инфраструктуры:** Выявлен нетипично высокий показатель среднего размера пожара для категории "Powerline" (45 гектаров — 2-е место после молний). Это формирует гипотезу о высокой сложности и длительности ликвидации возгораний на объектах энергетической инфраструктуры.
#### 4. **Оценка качества данных:** Наличие значительной доли категорий "Miscellaneous" и "Missing/Undefined" как в топе по частоте, так и в топе по суммарному ущербу, указывает на несовершенство системы классификации или логирования инцидентов в ранние периоды наблюдений.

#### **Итог:** Анализ показал, что превентивные меры по борьбе с лесными пожарами должны разделяться на две ветви: просветительская и административная работа с населением для снижения количества пожаров, и развитие систем раннего обнаружения в труднодоступных районах для минимизации колоссального ущерба от природных факторов.

## **Блок 7. Анализ длительности пожаров.**

### Необходимо исследовать временные характеристики пожаров. В данном блоке анализируется продолжительность возгораний, выявляются наиболее длительные пожары, а также исследуется связь между длительностью пожара, его причиной и местом возникновения.

### Какова средняя продолжительность пожара?

#### Перед расчетом среднего времени тушения необходимо оценить полноту данных и определить точный процент пропусков в колонке `СONT_DATETIME`. Высокая доля неопределенных значений NaN указывает на системное смещение выборки, так как данные обычно теряются либо для микро-инцидентов, либо для неуправляемых мега-пожаров. Без этого экспресс-анализа итоговое среднее арифметическое будет нерепрезентативным и исказит реальную картину.

In [41]:
fd["CONT_DATETIME"].isna().mean() * 100

np.float64(47.39371363089367)

#### Видим, что процент неопределенных значений достаточно высок, а значит, что просто определив среднее значение в общей выборке, мы получим результат, который отразит в себе только такие записи о пожарах, в которых указано время окончания горения. Поскольку данных мало и они смещены, среднее значение ломается из-за аномальных выбросов (мега-пожаров), а медиана устойчива к ним и показывает реальную картину для типичного пожара из всей генеральной совокупности, поэтому используем именно ее.

In [42]:
(fd["CONT_DATETIME"] - fd["DISCOVERY_DATETIME"]).median().round('min')

Timedelta('0 days 01:19:00')

#### Имеем, что половина пожаров с четко задокументированным временем окончания тушения горят менее 80 минут.

#### Посмотрим, как ситуация с пропусками обстоит относительно разных классов пожаров. Сделаем выборку с процентом неопределенных значений, а также со медианным временем тушения пожаров.

In [44]:
fd.groupby('FIRE_SIZE_CLASS').apply(lambda x: pd.Series({
    'MISSING_DATA_PCT': round(x['CONT_DATETIME'].isna().mean() * 100, 2),
    'MEDIAN_DURATION': (x['CONT_DATETIME'] - x['DISCOVERY_DATETIME']).median().round('min')
}), include_groups=False)

,MISSING_DATA_PCT,MEDIAN_DURATION
FIRE_SIZE_CLASS,,
A,36.74,0 days 01:00:00
B,54.23,0 days 01:09:00
C,52.91,0 days 02:46:00
D,44.70,0 days 12:04:00
E,36.09,1 days 05:16:00
F,25.48,3 days 10:03:00
G,14.63,13 days 09:02:00


#### Результат довольно неожиданный: оказывается, что во всех классах пожаров процент пропусков в данных об окончании тушения просто огромный. При этом видна четкая зависимость — чем серьезнее пожар, тем лучше его записывают. Если у мелких пожаров класса "B" пропусков больше половины (54%), то у катастрофических пожаров класса "G" их всего около 15%. Скорее всего, крупные инциденты проходят жесткий аудит на федеральном уровне, а на фиксацию времени мелких и рутинных пожаров службы тратят слишком мало времени из-за сложности заполнения всех ведомостей. С медианным временем тушения все предсказуемо: мелкие возгорания гасят за пару часов, а крупные пожары могут гореть днями или даже неделями.

### Какие штаты имеют самые продолжительные пожары? 

#### Для ответ на вопрос, определим, что продолжительными пожарами будут являться те, что длились более 3 дней.

In [53]:
fd['STATE'][fd["CONT_DATETIME"] - fd["DISCOVERY_DATETIME"] > pd.Timedelta(days=3)].value_counts().head(10)

STATE
WA    5311
AZ    4582
ID    4026
OR    3873
CA    3305
MT    3115
AK    3044
NM    2950
UT    2458
NY    2255
Name: count, dtype: int64

#### Результат достаточно интересный, ведь самые пожароопасные регионы вроде Калифорнии и Аляски находятся вообще не на первых местах. В топ вылезли совершенно другие штаты — Вашингтон "WA", Аризона "AZ" и Айдахо "ID". Скорее всего, дело просто в их климате и географии. На Северо-Западе и в Айдахо куча глухой тайги и гор, куда пожарной технике физически тяжело добраться. А Аляска "AK" не попала в лидеры потому, что в штате регистрируется мало пожаров по количеству, следовательно их не набралось много в этой выборке.

### Какие причины приводят к самым продолжительным пожарам?

#### Продолжительными пожарами пусть будем называть пожары, длящиеся больше 3 дней.

In [65]:
fd['STAT_CAUSE_DESCR'][fd["CONT_DATETIME"] - fd["DISCOVERY_DATETIME"] > pd.Timedelta(days=3)].value_counts()

STAT_CAUSE_DESCR
Lightning            28225
Miscellaneous         5888
Arson                 2999
Debris Burning        2231
Campfire              2065
Missing/Undefined     1854
Equipment Use         1331
Smoking                682
Children               559
Powerline              373
Fireworks              253
Railroad               181
Structure              118
Name: count, dtype: int64

#### Результат четко подтверждает мысль, что в самых долгих пожарах виновата именно природа. С огромным отрывом тут лидируют удары молний "Lightning" — аж 28 225 случаев, это примерно в 5 раз больше, чем у следующей причины. Логично, что такие пожары начинаются где-то в глухих лесах, где нет ни дорог, ни инфраструктуры. Из-за этого их замечают слишком поздно, и тушить их приходится намного дольше.

### Вывод по Блоку 7: Анализ длительности пожаров

#### В ходе исследования временных характеристик и продолжительности ликвидации возгораний были сформулированы следующие аналитические выводы:

#### 1. **Смещение данных и центральная тенденция:** Выявлен высокий уровень дефицита данных — в 47% записей отсутствует время ликвидации (`CONT_DATETIME`). В связи с высокой чувствительностью среднего арифметического к выбросам и пропускам, в качестве базовой метрики была выбрана медиана. Типичный (медианный) пожар в США ликвидируется за 1 час 19 минут, что говорит о высокой скорости реагирования на большинство штатных ситуаций.
#### 2. **Взаимосвязь масштаба и полноты логирования:** Обнаружена обратная зависимость между размером пожара и долей пропусков в данных. Для микро-пожаров класса А и B доля пропущенных меток времени составляет 36-54%, тогда как для мега-пожаров класса G она падает до 15%. Это свидетельствует о строгом протоколировании крупных экологических катастроф и упрощенном учете мелких инцидентов.
#### 3. **Географические очаги затяжных пожаров:** Наибольшее абсолютное количество пожаров длительностью более 3 дней зафиксировано в штатах Вашингтон (WA), Аризона (AZ) и Айдахо (ID). Это указывает на то, что сложный горно-лесистый ландшафт Северо-Запада и засушливый климат Юго-Запада являются главными факторами, препятствующими оперативной локализации огня.
#### 4. **Природный фактор как катализатор затяжного горения:** Абсолютным драйвером длительного горения выступают молнии -- на них приходится порядка 65% всех инцидентов, длящихся более 3 дней. В отличие от антропогенных факторов (поджоги, костры), которые происходят вблизи инфраструктуры, молнии бьют вглубь дикой территории, делая такие пожары самыми сложными для тушения.

## **Блок 8. Итоговый результат исследования.**

### **1. Краткое описание исследования.**

#### В ходе проекта был проведен исследовательский анализ датасета по лесным пожарам в США. Были очищены данные и изучены закономерности того, где, когда и почему возникают возгорания. Особое внимание уделялось тому, как заполнялись данные о времени тушения, и какие причины влияют на размер и длительность пожаров.

### **2. Основные результаты исследования.**

#### * Наибольшее количество пожаров пришлось на штаты Калифорния, Джорджия и Техас.
#### * По суммарной площади выгорания лидируют совершенно другие штаты, а именно Аляска, Айдахо и Калифорния.
#### * Самыми распространенными причинами пожаров оказались те, в которых задействованы люди, в первую очередь это сжигание отходов "Debris Burning" и поджоги "Arson".
#### * Самые крупные пожары (как по средней, так и по общей площади) чаще всего возникали из-за молний "Lightning". Восемь из десяти самых гигантских пожаров в датасете произошли именно по этой причине.
#### * Половина всех пожаров тушится быстрее 80 минут. При этом самые длительные пожары (более 3 дней) чаще всего наблюдались в штатах Вашингтон, Аризона и Айдахо, а их главной причиной тоже является "Lightning".

### **3. Интересные наблюдения.**

#### * Можно заметить определенный парадокс: люди устраивают абсолютное большинство пожаров, но они мелкие и быстро тушатся. В то же время природный фактор в виде молний приводит к относительно редким, но самым разрушительным последствиям.
#### * Результат по штату Аляска достаточно необычный: по количеству возгораний этот штат далеко не на первом месте, но по суммарной уничтоженной площади он обходит всех остальных.
#### * Значения у причины "Powerline" (линии электропередач) на первый взгляд неожиданные: по среднему размеру пожара эта причина на втором месте после молний. Скорее всего, такие пожары очень трудны для тушения.
#### * Процент пропусков в данных о времени тушения напрямую зависит от размера пожара. У мелких пожаров время не записывают больше чем в половине случаев, а у огромных пожаров класса G пропусков всего около 14%. То есть крупные катастрофы логируются пожарными службами гораздо лучше.

### **4. Ограничения исследования.**

#### * В датасете достаточно высокий процент неопределенных значений. Почти у половины записей отсутствует время окончания горения, поэтому считать обычное среднее было нельзя, пришлось использовать медиану.
#### * Большое число пожаров отнесено к категориям "Miscellaneous" (прочие) и "Missing/Undefined" (неизвестно). Это портит статистику, так как мы не знаем реальную причину возникновения огня.
#### * В данных отсутствуют погодные условия (температура, уровень осадков, сила ветра), хотя они очень сильно влияют на распространение огня.
#### * Отсутствует информация о рельефе местности, поэтому мы можем только догадываться, что в некоторых штатах пожары длятся дольше из-за глухих лесов и гор.

### **5. Что можно исследовать дальше.**

#### * Добавить погодные данные, чтобы исследовать влияние жары и долгих засух на количество пожаров.
#### * Выполнить пространственный анализ и нанести пожары на реальную карту, чтобы найти самые опасные очаги внутри штатов.
#### * Попробовать обучить модель прогнозирования, чтобы предсказывать класс размера пожара по штату, месяцу и причине.
#### * Найти данные о плотности населения и стоимости недвижимости, чтобы оценить примерный финансовый ущерб от возгораний.

### **6. Заключение.**

#### В результате выполнения проекта был проведен полный цикл исследовательского анализа данных: от загрузки и очистки до получения конкретных выводов. Для анализа активно использовались возможности библиотеки pandas, включая фильтрацию, группировку, работу с пропущенными значениями и вычисление статистических показателей. Полученные результаты позволяют получить общее представление о закономерностях возникновения лесных пожаров в США и демонстрируют мои практические навыки анализа данных.